# Projekt Zespołowy 2 — Zespół 10, "Sieci"

In [10]:
from net_opt.core.constraints.bandwidth_limit import BandwidthLimit
from net_opt.core.constraints.demand import Demand
from net_opt.core.constraints.dependent_demand import DependentDemand
from net_opt.core.constraints.path_bandwidth_io_match import PathBandwidthIOMatch
from net_opt.core.constraints.path_edge_bandwidth_kirchhoff import PathEdgeBandwidthKirchhoff
from net_opt.core.constraints.path_transponder_bandwidth_at_least import PathTransponderBandwidthAtLeast
from net_opt.core.ea import FastEA
from net_opt.core.mutations.base_mutation import Mutation
from net_opt.core.mutations.gene_mutation import GeneMutation
from net_opt.core.mutations.uniform_crossover import UniformCrossover
from net_opt.core.selections.tournament_selection import TournamentSelection
from net_opt.core.termination_conditions.base_termination_condition import TerminationCondition
from net_opt.core.termination_conditions.time_limit import TimeLimit
from net_opt.utiils.visualisation import visualize_input_data, visualize_population_individual

from pathlib import PurePath
from amplpy import ampl_notebook # type: ignore
import networkx as nx
import matplotlib.pyplot as plt
import time

In [11]:
AMPL_Path = "../src/"

# Benchmark AMPL

In [12]:
ampl = ampl_notebook(
    modules=["cplex"],
    license_uuid="2ac42500-338f-4917-9b9e-8fdc1cc63912"
)
ampl.read(str(PurePath(f"{AMPL_Path}/AMPL/PZSP2.mod")))
ampl.read_data(str(PurePath(f"{AMPL_Path}/AMPL/PZSP2.dat")))
N = ampl.get_parameter("n").value()
ampl.option["solver"] = "cplex"

Exception: Failed to install modules.

In [ ]:
start = time.perf_counter()
ampl.solve()
AMPL_time = time.perf_counter() - start
assert ampl.solve_result == "solved"
print(f"AMPL Solved in {AMPL_time:f}s")

In [ ]:
print(f"Total cost: {ampl.get_objective("total_cost").value()}")
flow = ampl.get_variable("usage").get_values().to_pandas()
print(f"Flow matrix:\n{flow}")
encryption = ampl.get_variable("encrypted").get_values().to_pandas()
print(f"Encrypted edges:\n{encryption}")
bandwidth = ampl.get_data("{(i,j) in EDGES} sum {(k,l) in CONNECTIONS} (usage[i,j,k,l]+usage[j,i,k,l])").to_pandas()
bandwidth.rename(columns={bandwidth.columns[0]: "bandwidth"}, inplace=True)
print(f"Bandwidth usage:\n{bandwidth}")

G = nx.Graph()
G.add_nodes_from(range(1,N+1))
G.add_weighted_edges_from([(i,j,bandwidth.at[(i,j),"bandwidth"]) for i in range(1,N+1) for j in range(1,N+1) if i<j])
nx.set_edge_attributes(G, encryption.to_dict()['encrypted.val'], "encrypted")
print(list(G.edges))
print(nx.get_edge_attributes(G, "encrypted"))
colors = [edge[2] for edge in G.edges.data("encrypted")]
pos = nx.shell_layout(G)
nx.draw(G, pos=pos, with_labels=True, font_weight='bold', edge_color=colors)
nx.draw_networkx_edge_labels(G, pos=pos, edge_labels=nx.get_edge_attributes(G, "weight"))
plt.show()

# Algorytm Ewolucyjny (WIP)

In [ ]:
import torch
from torch.distributions import Categorical
import pprint
pp = pprint.PrettyPrinter(indent=4)

## CUDA

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"--- Running on {device} ---")
torch.set_default_device(device)

## Experiment setup

In [ ]:
max_bandwidth=96
N = 20
T = 5 
mock_neigh_matrix = torch.triu(torch.ones(N, N), diagonal=1).bool()
# mock_demand = torch.triu(torch.randint(50, 150, (N, N)).float(), diagonal=1)
mock_demand = torch.triu(torch.full((N, N), 44000, dtype=torch.float))
mock_transponder_capacities = torch.tensor([40.0, 100.0, 200.0, 400.0, 800.0])
mock_transponder_costs = torch.tensor([1.0, 3.0, 5.0, 9.0, 11.0])

path_dist = Categorical(torch.ones(max(max_bandwidth//(N*N), 2)))
trans_dist = Categorical(torch.ones(max_bandwidth//T))
# path_dist = Bernoulli(0.5)
# trans_dist = Bernoulli(0.5)

mutation_config: list[Mutation] =[
    GeneMutation(
        path_edge_bandwidth_usage_mut_proba=0.5,
        path_transponder_assignment_mut_proba=0.5
    ),
    UniformCrossover(
        path_edge_bandwidth_usage_cross_proba=0.5,
        path_transponder_assignment_cross_proba=0.5
    )
]
selection_config = TournamentSelection(k=3)
termination_config : list[TerminationCondition] = [
    # MaxIterations(iterations=10), 
    # MinImprovement(delta=0.0001),
    TimeLimit(time_limit_s=AMPL_time)
]

ea_config = FastEA(
    show_vizualisation_every_n_iter=5000,
    population_size=1000,
    elite_size=50,
    path_edge_bandwidth_usage_init_distribution=path_dist,
    path_transponder_assignment_init_distribution=trans_dist,
    constraints=[
        PathBandwidthIOMatch(),
        PathEdgeBandwidthKirchhoff(),
        PathTransponderBandwidthAtLeast(),
        Demand(),
        # DependentDemand(),
        BandwidthLimit(),
        ],
    termination_conditions=termination_config,
    selection_method=selection_config,
    mutation_methods=mutation_config,
    global_constraint_weight=10000
)

## Visualize input

In [ ]:
visualize_input_data(
    mock_neigh_matrix,
    mock_demand,
    mock_transponder_costs,
    mock_transponder_capacities
)

## Experiment run

In [ ]:
ea_instance = ea_config
ea_instance.run(
    neigh_matrix=mock_neigh_matrix,
    demand=mock_demand,
    transponder_costs=mock_transponder_costs,
    transponder_capacities=mock_transponder_capacities
)

print("\n--- EA Run Finished ---")
print(f"Total iterations: {ea_instance._iteration_n}")
visualize_population_individual(ea_instance._population, ea_instance._transponder_capacities)
dict_constraint_name_scores ={
ea_instance.constraints[i].readable_name: ea_instance._lowest_penalty_constraint_scores[i]
for i in range(len(ea_instance._constraint_scores))
}
print("-"*40)
print(f"it: {ea_instance._iteration_n}")
print(f"lowest_penalty: {ea_instance._lowest_penalty}")
print("lowest_penalty_constraint_scores:")
pp.pprint(dict_constraint_name_scores) 
print(f"lowest_transponder_cost: {ea_instance._lowest_transponder_cost}")